# Collections Data Analyst Assignment
Analysis of the supplied collections dataset. I started with data checks before using the tables for the KPI calculations.

In [1]:
import pandas as pd, numpy as np
from pathlib import Path
BASE=Path('/mnt/data/collections_assignment')
def rd(n): return pd.read_csv(BASE/f'{n}.csv')
accounts=rd('accounts'); borrowers=rd('borrowers'); payments=rd('payments'); targeting=rd('daily_targeting'); calls=rd('calls'); sessions=rd('agent_sessions'); campaigns=rd('campaigns'); ptp=rd('promises_to_pay')

## 1. Data inventory and quality

In [2]:
inventory=[]
for p in BASE.glob('*.csv'):
    if p.name=='data_dictionary.csv': continue
    d=pd.read_csv(p); inventory.append([p.name,len(d),len(d.columns),int(d.duplicated().sum())])
pd.DataFrame(inventory,columns=['table','rows','cols','exact_duplicates']).sort_values('table')

,table,rows,cols,exact_duplicates
13,account_status_history.csv,60000,8,0
6,accounts.csv,30000,11,0
10,agent_sessions.csv,15000,7,0
5,agents.csv,30000,8,0
15,borrowers.csv,30600,8,600
7,call_attempts.csv,120000,9,0
2,call_dispositions.csv,35000,8,0
3,calls.csv,91350,11,1271
0,campaigns.csv,120,7,0
11,complaints.csv,8000,9,0


In [3]:
print('Borrower rows:',len(borrowers))
print('Duplicate borrower IDs:',borrowers.borrower_id.duplicated(False).sum())
print('Exact borrower duplicates:',borrowers.duplicated().sum())
counts=borrowers.groupby('borrower_id').size()
print('Accounts with unique/ambiguous/missing/invalid borrower links:', accounts.borrower_id.map(counts).eq(1).sum(), accounts.borrower_id.map(counts).gt(1).sum(), accounts.borrower_id.isna().sum(), accounts.borrower_id.notna().sum()-accounts.borrower_id.isin(counts.index).sum())

Borrower rows: 30600
Duplicate borrower IDs: 28151
Exact borrower duplicates: 600
Accounts with unique/ambiguous/missing/invalid borrower links: 5946 21141 455 2458


## 2. Golden payment cleaning

In [4]:
payments['exact_dup']=payments.duplicated(['account_id','borrower_id','event_at','payment_reference','amount','payment_status','payment_method','provider_id'])
golden=payments.loc[~payments.exact_dup].copy()
golden['month']=pd.to_datetime(golden.event_at).dt.to_period('M').astype(str)
success=golden[golden.payment_status.eq('SUCCESS')]
monthly=success.groupby('month').agg(cash_recovered=('amount','sum'),recovered_accounts=('account_id','nunique'),transactions=('payment_id','nunique'))
monthly['cash_mom_pct']=monthly.cash_recovered.pct_change()*100
monthly

,cash_recovered,recovered_accounts,transactions,cash_mom_pct
month,,,,
2026-01,1.872291e+08,2374,2464,NaN
2026-02,1.702796e+08,2173,2268,-9.052818
2026-03,1.891903e+08,2419,2524,11.105681
2026-04,1.752289e+08,2304,2406,-7.379551
2026-05,1.843355e+08,2344,2449,5.196946
2026-06,1.758534e+08,2286,2366,-4.601411
2026-07,1.872478e+08,2335,2441,6.479478
2026-08,4.710970e+07,608,616,-74.840993


## 3. Is the 11% claim real?

In [5]:
t=targeting.copy(); t['month']=pd.to_datetime(t.target_date).dt.to_period('M').astype(str)
target_base=t.groupby('month').account_id.nunique().rename('targeted_accounts')
monthly=monthly.join(target_base)
monthly['recovery_account_rate']=monthly.recovered_accounts/monthly.targeted_accounts
monthly['recovery_per_targeted_account']=monthly.cash_recovered/monthly.targeted_accounts
monthly.loc[['2026-02','2026-03']].assign(rate_mom=monthly.recovery_account_rate.pct_change()*100)

,cash_recovered,recovered_accounts,transactions,cash_mom_pct,targeted_accounts,recovery_account_rate,recovery_per_targeted_account,rate_mom
month,,,,,,,,
2026-02,1.702796e+08,2173,2268,-9.052818,5160,0.421124,32999.925382,1.679989
2026-03,1.891903e+08,2419,2524,11.105681,5666,0.426933,33390.456509,1.379297


**Interpretation:** cash recovered rises by about 11% from February to March, but recovery rate per targeted account changes only slightly. I would not use the 11% figure as the main measure of improvement.

## 4. Targeting and attribution

In [6]:
t=targeting.copy(); t['dt']=pd.to_datetime(t.target_date); t=t.merge(campaigns[['campaign_id','target_definition','strategy_version','channel']],on='campaign_id')
pt=ptp.copy(); pt['broken_dt']=pd.to_datetime(pt.event_at); pt=pt[pt.status.eq('BROKEN')][['account_id','broken_dt']].sort_values('broken_dt')
q=t[['target_id','account_id','dt','target_definition','recommended_channel']].sort_values('dt')
z=pd.merge_asof(q,pt,left_on='dt',right_on='broken_dt',by='account_id',direction='backward',tolerance=pd.Timedelta(days=365))
print('PROMISE_BROKEN targets:',(q.target_definition=='PROMISE_BROKEN').sum())
print('with prior broken PTP:',z.loc[q.target_definition.eq('PROMISE_BROKEN'),'broken_dt'].notna().sum())

PROMISE_BROKEN targets: 9021
with prior broken PTP: 650


## 5. Counterfactual design

The strategy versions overlap during the period, so a simple before/after comparison would mix the strategy effect with other changes. I would test the new strategy with randomized treatment and control groups within DPD, risk and loan-type groups. The main outcome would be 30-day net recovered ₹ per eligible account, with a 10% holdout.

## 6. Investment conclusion

I would start with **better borrower targeting**, but as a stage-gated experiment. The targeting data has a clear quality issue, while the dataset does not provide enough cost or causal uplift information to claim a positive ₹10 Cr ROI.